# Document Question Answering System (RAG)

**Week 7 Project — Ishita Tripathi**

A Retrieval-Augmented Generation (RAG) system that answers questions grounded in a custom PDF document, using:
- **Cohere** for embeddings, generation, and reranking
- **Pinecone** as the vector database
- **PyMuPDF** for PDF text extraction

## Architecture

```
PDF -> Text Extraction -> Chunking -> Embedding (Cohere) -> Pinecone Index
                                                                  |
User Question -> Query Embedding -> Retrieve top-k -> Rerank ----+
                                                                  |
                                    Retrieved Chunks -> Cohere Chat (grounded generation) -> Answer
```

Run the cells below in order.

## 1. Install dependencies

In [ ]:
!pip install cohere pinecone-client PyMuPDF -q

## 2. Imports

In [ ]:
import fitz  # PyMuPDF
import cohere
from pinecone import Pinecone, ServerlessSpec

## 3. Enter your API keys

- Get a free Cohere key: https://dashboard.cohere.com/api-keys
- Get a free Pinecone key: https://app.pinecone.io

In [ ]:
COHERE_API_KEY = input("Enter your Cohere API key: ")
PINECONE_API_KEY = input("Enter your Pinecone API key: ")

## 4. VectorStore class

Handles the retrieval half of the pipeline:
1. Load a PDF and extract raw text
2. Split the text into overlapping chunks
3. Embed each chunk with Cohere
4. Store/index the embeddings in Pinecone
5. Retrieve + rerank the most relevant chunks for a given query

In [ ]:
class VectorStore:
    def __init__(self, pdf_path: str, cohere_api_key: str, pinecone_api_key: str,
                 index_name: str = "rag-qa-bot", retrieve_top_k: int = 10, rerank_top_k: int = 3):
        self.pdf_path = pdf_path
        self.co = cohere.Client(cohere_api_key)
        self.pinecone_api_key = pinecone_api_key
        self.index_name = index_name
        self.retrieve_top_k = retrieve_top_k
        self.rerank_top_k = rerank_top_k

        self.chunks = []
        self.embeddings = []

        # Build the pipeline end-to-end on init
        self.pdf_text = self._extract_text_from_pdf(self.pdf_path)
        self._split_text()
        self._embed_chunks()
        self._index_chunks()

    # ---------- Document Ingestion ----------
    def _extract_text_from_pdf(self, pdf_path: str) -> str:
        text = ""
        with fitz.open(pdf_path) as pdf:
            for page_num in range(pdf.page_count):
                page = pdf.load_page(page_num)
                text += page.get_text("text")
        return text

    # ---------- Text Chunking ----------
    def _split_text(self, chunk_size: int = 1000):
        """Naive sentence-based chunking, capped at chunk_size characters."""
        sentences = self.pdf_text.split(". ")
        current_chunk = ""
        for sentence in sentences:
            if len(current_chunk) + len(sentence) < chunk_size:
                current_chunk += sentence + ". "
            else:
                self.chunks.append(current_chunk.strip())
                current_chunk = sentence + ". "
        if current_chunk.strip():
            self.chunks.append(current_chunk.strip())

    # ---------- Embedding Creation ----------
    def _embed_chunks(self, batch_size: int = 90):
        total_chunks = len(self.chunks)
        for i in range(0, total_chunks, batch_size):
            batch = self.chunks[i:min(i + batch_size, total_chunks)]
            batch_embeddings = self.co.embed(
                texts=batch,
                input_type="search_document",
                model="embed-english-v3.0",
            ).embeddings
            self.embeddings.extend(batch_embeddings)

    # ---------- Vector Database ----------
    def _index_chunks(self):
        pc = Pinecone(api_key=self.pinecone_api_key)

        if self.index_name not in pc.list_indexes().names():
            pc.create_index(
                name=self.index_name,
                dimension=len(self.embeddings[0]),
                metric="cosine",
                spec=ServerlessSpec(cloud="aws", region="us-east-1"),
            )
        self.index = pc.Index(self.index_name)

        chunks_metadata = [{"text": chunk} for chunk in self.chunks]
        ids = [str(i) for i in range(len(self.chunks))]
        self.index.upsert(vectors=zip(ids, self.embeddings, chunks_metadata))

    # ---------- Query Processing + Context Retrieval ----------
    def retrieve(self, query: str) -> list:
        query_emb = self.co.embed(
            texts=[query],
            model="embed-english-v3.0",
            input_type="search_query",
        ).embeddings[0]  # embed() always returns a list of vectors; take the single query vector

        res = self.index.query(
            vector=query_emb,
            top_k=self.retrieve_top_k,
            include_metadata=True,
        )

        docs_to_rerank = [match["metadata"]["text"] for match in res["matches"]]
        if not docs_to_rerank:
            return []

        rerank_results = self.co.rerank(
            query=query,
            documents=docs_to_rerank,
            top_n=min(self.rerank_top_k, len(docs_to_rerank)),
            model="rerank-v3.5",
        )
        return [res["matches"][result.index]["metadata"] for result in rerank_results.results]

## 5. Chatbot class

Handles the augmentation + generation half of the pipeline, using Cohere's current Chat V2 API.

In [ ]:
class Chatbot:
    def __init__(self, vectorstore, cohere_api_key: str, model: str = "command-r-08-2024"):
        self.vectorstore = vectorstore
        self.co = cohere.ClientV2(cohere_api_key)
        self.model = model
        self.messages = [
            {
                "role": "system",
                "content": "You are a helpful assistant that answers questions "
                            "using only the provided document context. If the "
                            "context doesn't contain the answer, say so.",
            }
        ]

    def respond(self, user_message: str):
        # ---------- Context Retrieval ----------
        retrieved_docs = self.vectorstore.retrieve(user_message)
        documents = [{"data": {"text": d.get("text", "")}} for d in retrieved_docs] or None

        self.messages.append({"role": "user", "content": user_message})

        # ---------- Answer Generation (grounded) ----------
        response = self.co.chat_stream(
            model=self.model,
            messages=self.messages,
            documents=documents,
        )

        return response, retrieved_docs

    def record_assistant_turn(self, full_text: str):
        """Call this after streaming completes, to keep multi-turn context."""
        self.messages.append({"role": "assistant", "content": full_text})

## 6. Upload your PDF

In [ ]:
from google.colab import files  # remove this line if not running on Colab

print("Upload the PDF you want to ask questions about:")
uploaded = files.upload()
pdf_filename = list(uploaded.keys())[0]
print(f"Uploaded: {pdf_filename}")

## 7. Build the vector store

This extracts text, chunks it, embeds each chunk, and indexes everything in Pinecone. Takes a bit longer the first time (creating the index).

In [ ]:
vs = VectorStore(pdf_filename, cohere_api_key=COHERE_API_KEY, pinecone_api_key=PINECONE_API_KEY)

extracted_chars = len(vs.pdf_text.strip())
print(f"Extracted {extracted_chars} characters of text.")
print(f"Indexed {len(vs.chunks)} chunks.")

if extracted_chars < 50:
    print("\nWarning: very little text was extracted. This usually means the PDF "
          "stores text as images/design elements rather than real selectable text "
          "(common with Canva-style templates). Try a different PDF if answers look empty.")

## 8. Create the chatbot and ask a question

In [ ]:
bot = Chatbot(vs, cohere_api_key=COHERE_API_KEY)

def ask(question: str):
    response, sources = bot.respond(question)

    print("Answer:\n")
    accumulated = ""
    for event in response:
        if event.type == "content-delta":
            text = event.delta.message.content.text
            accumulated += text
            print(text, end="")

    bot.record_assistant_turn(accumulated)
    print("\n")
    return sources

In [ ]:
sources = ask("What is the main idea of the document?")

## 9. (Optional) Inspect which chunks were used

In [ ]:
for i, s in enumerate(sources, 1):
    print(f"--- Source {i} ---")
    print(s.get("text", "")[:400], "...\n")

## 10. Ask more questions

The chatbot keeps conversation history, so you can keep calling `ask(...)` with follow-up questions.

In [ ]:
ask("Summarize the key points in one paragraph")

## Key Concepts

- **Retrieval** — finding the most relevant chunks of text from a document using embeddings and vector similarity search.
- **Augmentation** — adding retrieved content to the model's input to provide context for answering.
- **Generation** — the language model generates the final answer using retrieved context, keeping responses grounded in actual data.

## Key Learnings

- How RAG systems combine retrieval and generation
- Importance of retrieval in improving answer accuracy
- Working with embeddings and vector databases
- Handling unstructured text data
- Designing scalable AI pipelines

## Possible Improvements

- Better chunking strategies (semantic/recursive chunking instead of naive sentence splitting)
- Try different embedding models
- Hybrid search (keyword + vector)
- Additional re-ranking tuning
- Experiment with different generation models